# Academic PDF Structure Extraction (test4)

This notebook reads an academic PDF and performs:

- Chapter detection
- Topic detection
- Start/end page calculation
- JSON schema generation
- Folder structure creation
- Optional Docling-based structural conversion

The extracted structure is saved under `Backend/Experiments/test4_output`.

This notebook uses Docling when available to improve structure extraction, but still falls back to layout/font-size heuristics if Docling is unavailable.

For academic books, regex alone is usually the wrong approach because:

- Different PDFs use different formatting.
- Chapter titles may not follow a fixed pattern.
- Topics may span multiple lines.
- Some books use Roman numerals, others use numbered chapters, and some use visual-only headings.
- PDF text extraction often destroys original layout and formatting.

Docling is used here because it can preserve higher-level document structure that PDF text extraction alone often loses, especially for academic books where headings and sections are not consistently encoded.


In [1]:
from pathlib import Path
import json
import os
import re
from dotenv import load_dotenv
import fitz

# 1. find repository root by walking up until .git exists
repo_root = Path.cwd()
while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")
pdf_path = os.getenv("PDF_PATH")
if not pdf_path:
    raise ValueError("PDF_PATH must be set in .env")
pdf_path = Path(pdf_path)
if not pdf_path.is_absolute():
    pdf_path = repo_root / pdf_path
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF file not found: {pdf_path}")

doc = fitz.open(str(pdf_path))
print("PDF path:", pdf_path)
print("Total pages:", len(doc))

output_dir = repo_root / "Backend" / "Experiments" / "test4_output"
output_dir.mkdir(parents=True, exist_ok=True)
structure_path = output_dir / "structure_schema.json"
folder_root = output_dir / "Output"
toc_path = output_dir / "pdf_toc.json"
print("Output directory:", output_dir)


PDF path: c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\Backend\Books\sample_book.pdf
Total pages: 1170
Output directory: c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\Backend\Experiments\test4_output


In [3]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

result = converter.convert(
    str(pdf_path),
    page_range=(1, 5)
)

markdown = result.document.export_to_markdown()

print(markdown[:1000])

[INFO] 2026-06-25 13:10:22,725 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-25 13:10:22,854 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-25 13:10:23,011 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-25 13:10:23,017 [RapidOCR] main.py:50: Using C:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-25 13:10:24,169 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-25 13:10:24,175 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-25 13:10:24,188 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-25 13:10:24,191 [Rapi

## OR O +

## Clinically Oriented ANATOMY

Seventh Edition

Keith L. Moore Arthur F. Dalley Anne M.R. Agur

<!-- image -->

Healih Introduction

<!-- image -->

1  Thorax

2  Abdomen

3  Pelvis and Perineum

4  Back

5  Lower Limb

6  Upper Limb

7  Head

8  Neck

9  Cranial Nerves


## Task 1: Read PDF and find structure

This cell extracts heading candidates from the PDF using font-size and layout heuristics.

In [ ]:
from pathlib import Path
import json
import os
from dotenv import load_dotenv
import fitz

# --------------------------------------------------
# FIND REPO ROOT
# --------------------------------------------------

repo_root = Path.cwd()

while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

# --------------------------------------------------
# LOAD PDF PATH
# --------------------------------------------------

load_dotenv(repo_root / ".env")

pdf_path = os.getenv("PDF_PATH")

if not pdf_path:
    raise ValueError("PDF_PATH not found")

pdf_path = Path(pdf_path)

if not pdf_path.is_absolute():
    pdf_path = repo_root / pdf_path

if not pdf_path.exists():
    raise FileNotFoundError(pdf_path)

# --------------------------------------------------
# OPEN PDF
# --------------------------------------------------

doc = fitz.open(str(pdf_path))

print("=" * 50)
print("PDF:", pdf_path)
print("TOTAL PAGES:", len(doc))
print("=" * 50)

# --------------------------------------------------
# OUTPUT FOLDER
# --------------------------------------------------

output_dir = repo_root / "Output"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------------
# EXTRACT PAGE STRUCTURE
# --------------------------------------------------

structured_pages = []

for page_number in range(len(doc)):

    page = doc[page_number]

    page_data = {
        "page": page_number + 1,
        "elements": []
    }

    blocks = page.get_text("dict")["blocks"]

    for block in blocks:

        if "lines" not in block:
            continue

        for line in block["lines"]:

            text = ""

            max_font = 0

            for span in line["spans"]:

                text += span["text"]

                max_font = max(
                    max_font,
                    span["size"]
                )

            text = text.strip()

            if text:

                page_data["elements"].append(
                    {
                        "text": text,
                        "font_size": round(
                            max_font,
                            2
                        )
                    }
                )

    structured_pages.append(page_data)

# --------------------------------------------------
# SAVE STRUCTURE JSON
# --------------------------------------------------

structure_file = output_dir / "structure_schema.json"

with open(
    structure_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        structured_pages,
        f,
        indent=4,
        ensure_ascii=False
    )

print()
print("Saved:", structure_file)

# --------------------------------------------------
# PRINT FIRST PAGE
# --------------------------------------------------

print("\nFIRST PAGE SAMPLE\n")

for item in structured_pages[0]["elements"][:30]:

    print(
        f"{item['font_size']:>5} | "
        f"{item['text']}"
    )

# --------------------------------------------------
# HEADING CANDIDATES
# --------------------------------------------------

heading_candidates = []

for page in structured_pages:

    for item in page["elements"]:

        if item["font_size"] >= 14:

            heading_candidates.append(
                {
                    "page": page["page"],
                    "text": item["text"],
                    "font_size": item["font_size"]
                }
            )

heading_file = output_dir / "heading_candidates.json"

with open(
    heading_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        heading_candidates,
        f,
        indent=4,
        ensure_ascii=False
    )

print()
print("Saved:", heading_file)

print("\nFIRST 50 HEADINGS\n")

for h in heading_candidates[:500]:

    print(
        f"PAGE {h['page']:>3} | "
        f"{h['font_size']:>5} | "
        f"{h['text']}"
    )

In [ ]:
from pathlib import Path
import json
import os

from dotenv import load_dotenv
import fitz

repo_root = Path.cwd()
while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

pdf_path = os.getenv("PDF_PATH")
if not pdf_path:
    raise ValueError("PDF_PATH must be set in .env")

pdf_path = Path(pdf_path)
if not pdf_path.is_absolute():
    pdf_path = repo_root / pdf_path

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF file not found: {pdf_path}")

doc = fitz.open(str(pdf_path))

output_dir = repo_root / "Backend" / "Experiments" / "test4_output"
output_dir.mkdir(parents=True, exist_ok=True)

dump_path = output_dir / "dump.json"
structure_path = output_dir / "structure_schema.json"
headings_path = output_dir / "heading_candidates.json"
md_file = output_dir / "book.md"

print("PDF path:", pdf_path)
print("Total pages:", len(doc))
print("Output directory:", output_dir)

In [ ]:
page_text_dump = []

for page_num in range(len(doc)):
    page = doc[page_num]
    page_text_dump.append(
        {
            "page": page_num + 1,
            "text": page.get_text("text").strip(),
        }
    )

with open(dump_path, "w", encoding="utf-8") as f:
    json.dump(page_text_dump, f, indent=4, ensure_ascii=False)

print("Saved:", dump_path)
print("Pages saved:", len(page_text_dump))

In [ ]:
print("\nFIRST NON-EMPTY PAGE TEXT SAMPLE\n")

sample_page = next((p for p in page_text_dump if p["text"]), None)
if sample_page:
    print("Page:", sample_page["page"])
    print(sample_page["text"][:2000])
else:
    print("No extractable text found in the PDF.")

In [ ]:
structured_pages = []

for page_num in range(len(doc)):
    page = doc[page_num]
    page_rect = page.rect

    page_data = {
        "page": page_num + 1,
        "width": round(page_rect.width, 2),
        "height": round(page_rect.height, 2),
        "elements": [],
    }

    blocks = page.get_text("dict")["blocks"]

    for block_index, block in enumerate(blocks):
        if "lines" not in block:
            continue

        for line_index, line in enumerate(block["lines"]):
            line_text = ""
            max_font_size = 0
            font_names = []
            bbox = line.get("bbox")

            for span in line["spans"]:
                line_text += span.get("text", "")
                max_font_size = max(max_font_size, span.get("size", 0))
                font_name = span.get("font")
                if font_name and font_name not in font_names:
                    font_names.append(font_name)

            line_text = " ".join(line_text.split())

            if line_text:
                page_data["elements"].append(
                    {
                        "text": line_text,
                        "font_size": round(max_font_size, 2),
                        "font_names": font_names,
                        "bbox": [round(value, 2) for value in bbox] if bbox else None,
                        "block_index": block_index,
                        "line_index": line_index,
                    }
                )

    structured_pages.append(page_data)

print("Structured pages:", len(structured_pages))
print("First page elements:", len(structured_pages[0]["elements"]) if structured_pages else 0)

In [ ]:
with open(structure_path, "w", encoding="utf-8") as f:
    json.dump(structured_pages, f, indent=4, ensure_ascii=False)

print("Saved:", structure_path)

In [ ]:
first_non_empty_page = next((p for p in structured_pages if p["elements"]), None)

if first_non_empty_page:
    print("First non-empty structured page:", first_non_empty_page["page"])
    print("Elements on that page:", len(first_non_empty_page["elements"]))
    print("\nFIRST PAGE\n")

    for item in first_non_empty_page["elements"][:30]:
        print(f"{item['font_size']:>5} | {item['text']}")
else:
    print("No structured text elements found in the PDF.")

In [ ]:
heading_candidates = []

for page in structured_pages:
    for item in page["elements"]:
        if item["font_size"] >= 14:
            heading_candidates.append(
                {
                    "page": page["page"],
                    "text": item["text"],
                    "font_size": item["font_size"],
                    "bbox": item["bbox"],
                }
            )

with open(headings_path, "w", encoding="utf-8") as f:
    json.dump(heading_candidates, f, indent=4, ensure_ascii=False)

print("Saved:", headings_path)
print("Total possible headings:", len(heading_candidates))

print("\nPOSSIBLE HEADINGS\n")
for h in heading_candidates[:100]:
    print(f"Page {h['page']} | {h['font_size']} | {h['text']}")

In [ ]:
# import sys

# print("Python:", sys.executable)

# try:
#     import torch
#     print("Torch file:", torch.__file__)
#     print("Torch version:", torch.__version__)
# except Exception as e:
#     print("Torch error:", e)

# from docling.document_converter import DocumentConverter

# print("Import OK")

# converter = DocumentConverter()

# print("Converter OK")

In [ ]:
# print("Loading Docling. On Windows this can take 1-2 minutes...")

# try:
#     from docling.document_converter import DocumentConverter

#     print("Converting PDF to Markdown. Large PDFs can take several minutes...")
#     converter = DocumentConverter()
#     result = converter.convert(str(pdf_path))
#     markdown = result.document.export_to_markdown()
# except OSError as exc:
#     if "c10.dll" in str(exc) or "dynamic link library" in str(exc).lower():
#         print("PyTorch DLL load failed. Restart the notebook kernel and select 'AI PDF Pipeline (venv)'.")
#     raise

# with open(md_file, "w", encoding="utf-8") as f:
#     f.write(markdown)

# print("Markdown saved:", md_file) 
# print("Markdown characters:", len(markdown))

print("Loading Docling...")

from docling.document_converter import DocumentConverter

converter = DocumentConverter()

result = converter.convert(pdf_path)

markdown = result.document.export_to_markdown()

with open(md_file, "w", encoding="utf-8") as f:
    f.write(markdown)

print("Markdown saved:", md_file)
print("Characters:", len(markdown))


In [ ]:
print("\nMARKDOWN SAMPLE\n")
print(markdown[:2000])

In [4]:
from pathlib import Path
import json
import os
from dotenv import load_dotenv
import fitz

# -----------------------------
# Locate Repository Root
# -----------------------------
repo_root = Path.cwd()

while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

pdf_path = os.getenv("PDF_PATH")

if not pdf_path:
    raise ValueError("PDF_PATH not found inside .env")

pdf_path = Path(pdf_path)

if not pdf_path.is_absolute():
    pdf_path = repo_root / pdf_path

if not pdf_path.exists():
    raise FileNotFoundError(pdf_path)

doc = fitz.open(pdf_path)

print(f"Total Pages : {len(doc)}")

# -----------------------------------
# Output Folder
# -----------------------------------

output_dir = repo_root / "Backend" / "Experiments" / "test3_output"
output_dir.mkdir(parents=True, exist_ok=True)
json_output = output_dir / "chapters_topics.json"

# -----------------------------------
# Extract TOC
# -----------------------------------

toc = doc.get_toc()

if len(toc) == 0:
    raise Exception(
        "This PDF does not contain bookmarks (TOC). "
        "Need another algorithm using font detection."
    )

# -----------------------------------
# DEBUG: Print raw TOC to understand levels
# -----------------------------------

print("\n--- RAW TOC (first 40 entries) ---")
for item in toc[:40]:
    level, title, page = item
    print(f"  Level {level} | Page {page:4d} | {'  ' * (level-1)}{title}")
print("--- END TOC PREVIEW ---\n")

# -----------------------------------
# Detect Chapter Level Dynamically
# -----------------------------------

import re

def is_chapter_title(title):
    """
    Matches titles like:
      '1 Thorax', '2 Abdomen', 'Chapter 1', 'Chapter 1 Thorax'
      'Introduction to Clinically Oriented Anatomy'  <- level 1 non-numbered
    We treat ALL level-1 TOC entries as chapters.
    """
    # Numbered chapter: starts with digit(s) optionally followed by text
    if re.match(r'^\d+[\s\.\-]', title.strip()):
        return True
    # Explicit word 'chapter'
    if 'chapter' in title.lower():
        return True
    return False


# -----------------------------------
# Build Chapter/Topic/Subtopic Tree
# -----------------------------------

chapters = []
current_chapter = None
current_topic = None

for i, item in enumerate(toc):

    level, title, start_page = item

    # Compute raw end page (next entry's start - 1)
    if i < len(toc) - 1:
        next_start = toc[i + 1][2]
        end_page = next_start - 1
    else:
        end_page = len(doc)

    if level == 1:
        # Every level-1 entry = a chapter (Introduction, Chapter 1, 2, 3 ...)
        current_chapter = {
            "chapter": title,
            "start_page": start_page,
            "end_page": end_page,       # will be fixed later
            "topics": []
        }
        current_topic = None
        chapters.append(current_chapter)

    elif level == 2 and current_chapter is not None:
        # Level-2 = main section / topic inside a chapter
        current_topic = {
            "topic": title,
            "start_page": start_page,
            "end_page": end_page,       # will be fixed later
            "subtopics": []
        }
        current_chapter["topics"].append(current_topic)

    elif level == 3 and current_topic is not None:
        # Level-3 = subtopic inside a topic
        current_topic["subtopics"].append({
            "subtopic": title,
            "start_page": start_page,
            "end_page": end_page
        })

    elif level == 3 and current_chapter is not None:
        # Edge case: level-3 appears but no current topic yet
        # Attach directly to chapter as a flat topic
        current_chapter["topics"].append({
            "topic": title,
            "start_page": start_page,
            "end_page": end_page,
            "subtopics": []
        })

# -----------------------------------
# Fix Chapter End Pages
# -----------------------------------

for i in range(len(chapters)):
    if i < len(chapters) - 1:
        chapters[i]["end_page"] = chapters[i + 1]["start_page"] - 1
    else:
        chapters[i]["end_page"] = len(doc)

# -----------------------------------
# Fix Topic End Pages within each Chapter
# -----------------------------------

for chapter in chapters:
    topics = chapter["topics"]
    for j in range(len(topics)):
        if j < len(topics) - 1:
            topics[j]["end_page"] = topics[j + 1]["start_page"] - 1
        else:
            topics[j]["end_page"] = chapter["end_page"]

    # Fix subtopic end pages within each topic
    for topic in topics:
        subtopics = topic["subtopics"]
        for k in range(len(subtopics)):
            if k < len(subtopics) - 1:
                subtopics[k]["end_page"] = subtopics[k + 1]["start_page"] - 1
            else:
                subtopics[k]["end_page"] = topic["end_page"]

# -----------------------------------
# Save JSON
# -----------------------------------

result = {
    "pdf_name": pdf_path.name,
    "total_pages": len(doc),
    "total_chapters": len(chapters),
    "chapters": chapters
}

with open(json_output, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)

print("JSON Saved ->", json_output)

# -----------------------------------
# Create Folder Structure
# -----------------------------------

root_folder = output_dir / "PDF_STRUCTURE"
root_folder.mkdir(exist_ok=True)

invalid_chars = '<>:"/\\|?*'

def clean(name):
    for c in invalid_chars:
        name = name.replace(c, "")
    return name.strip()[:80]   # also cap length to avoid OS path limits


for chapter in chapters:

    chapter_folder = root_folder / clean(chapter["chapter"])
    chapter_folder.mkdir(exist_ok=True)

    for topic in chapter["topics"]:

        print(f"  Topic: {topic['topic']}")
        topic_folder = chapter_folder / clean(topic["topic"])
        topic_folder.mkdir(exist_ok=True)

        for subtopic in topic.get("subtopics", []):
            print(f"    Subtopic: {subtopic['subtopic']}")
            subtopic_folder = topic_folder / clean(subtopic["subtopic"])
            

# print("\nFolder Structure Created Successfully")

# print("=" * 80)

Total Pages : 1170

--- RAW TOC (first 40 entries) ---
  Level 1 | Page    1 | Cover
  Level 1 | Page    5 | Title Page
  Level 1 | Page    6 | Copyright
  Level 1 | Page    7 | Dedication
  Level 1 | Page    8 | About the Authors
  Level 1 | Page    9 | Preface
  Level 2 | Page    9 |   KEY FEATURES
  Level 2 | Page   10 |   RETAINED AND IMPROVED FEATURES
  Level 2 | Page   11 |   COMMITMENT TO EDUCATING STUDENTS
  Level 2 | Page   11 |   ABBREVIATIONS
  Level 1 | Page   13 | Acknowledgments
  Level 1 | Page   17 | Contents
  Level 1 | Page   21 | List of Clinical Blue Boxes
  Level 1 | Page   27 | Figure Credits
  Level 1 | Page   31 | Introduction to Clinically Oriented Anatomy
  Level 2 | Page   32 |   APPROACHES TO STUDYING ANATOMY
  Level 3 | Page   32 |     Regional Anatomy
  Level 3 | Page   33 |     Systemic Anatomy
  Level 3 | Page   34 |     Clinical Anatomy
  Level 2 | Page   34 |   ANATOMICOMEDICAL TERMINOLOGY
  Level 3 | Page   35 |     Anatomical Position
  Level 3 | Pag